# Dataset v3: cross-speaker validation of horizontal flip

This notebook checks whether horizontal-flip augmentation helps the frozen
LRW ResNet18 + BiGRU model on several unseen speakers, rather than only on
the fixed `spk06` validation set.

The final test speakers are never loaded or used here.

## 1. Подключить Google Drive и получить зависимости

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

## 2. Скачать официальный LRW-код и найти dataset v3

In [ ]:
from pathlib import Path
import copy
import random
import shutil
import subprocess
import sys
import time
import zipfile

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
upstream_store = drive_root / 'upstream_lrw_pretrained'
upstream_store.mkdir(parents=True, exist_ok=True)

if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)

checkpoint_path = upstream_store / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    downloaded = gdown.download(
        id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD',
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

extract_dir = Path('/content/lipreading_data_v3')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_landmark_dataset_v3'
    / 'speaker_top10'
)
expected_split_dir = extract_dir / 'ru_dataset' / relative_split_dir
if not expected_split_dir.exists():
    matches = list(Path('/content/drive').rglob(
        'colab_lipreading_dataset_v3_train_val.zip'
    ))
    if not matches:
        raise FileNotFoundError(
            'Upload colab_lipreading_dataset_v3_train_val.zip to Google Drive.'
        )
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0], 'r') as archive:
        archive.extractall(extract_dir)
    print('Dataset restored from:', matches[0])

sys.path.insert(0, str(upstream_dir))
data_root = extract_dir / 'ru_dataset'
split_dir = data_root / relative_split_dir
output_dir = drive_root / 'lrw_frozen_hflip_cross_speaker_dataset_v3'
output_dir.mkdir(parents=True, exist_ok=True)
print('dataset:', data_root)
print('checkpoint:', checkpoint_path)
print('output:', output_dir)

## 3. Прочитать только train и validation

Test намеренно не загружается: по нему нельзя подбирать архитектуру или learning rate.

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('Enable a GPU runtime before running this experiment.')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}

assert len(train_df) == 1901
assert len(val_df) == 105
assert set(train_df['speaker_id']).isdisjoint(set(val_df['speaker_id']))
assert all((data_root / path).is_file() for path in train_df['clip_path'])
assert all((data_root / path).is_file() for path in val_df['clip_path'])

print('device:', device)
print('train:', len(train_df), sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), sorted(val_df['speaker_id'].unique()))
print('vocab:', vocab)
print('TEST SPLIT IS NOT LOADED.')

## 4. Датасет видеоклипов

Вход модели: 24 grayscale-кадра, `88x88`, с LRW-нормализацией. Здесь аугментаций нет, чтобы измерить только эффект partial fine-tuning.

In [ ]:
def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frames = []
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        offset = (input_size - crop_size) // 2
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()

    if not frames:
        raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)


class EndToEndVideoDataset(Dataset):
    def __init__(self, metadata, data_root):
        self.metadata = metadata.reset_index(drop=True)
        self.data_root = data_root

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        video = load_lrw_video(self.data_root / row['clip_path'])
        label = word_to_idx[row['word']]
        return video, label


def make_video_loaders(seed, batch_size=8):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        EndToEndVideoDataset(train_df, data_root),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        EndToEndVideoDataset(val_df, data_root),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_loader, val_loader

preview_train_loader, _ = make_video_loaders(seed=42)
sample_x, sample_y = next(iter(preview_train_loader))
print('input:', sample_x.shape, sample_x.min().item(), sample_x.max().item())
print('labels:', sample_y.tolist())

## 5. LRW encoder + BiGRU

Сначала загружаются официальные LRW-веса. Затем frozen baseline с `uniform shuffle` и seed 42 точно воспроизводится на сохранённых LRW-признаках. Его лучшие веса BiGRU становятся стартом для partial fine-tuning.

In [ ]:
from lipreading.model import Lipreading


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


class PartialFineTuneLipReader(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, videos):
        features = self.encoder(
            videos, lengths=[videos.shape[2]] * videos.shape[0]
        )
        return self.head(features)


tcn_options = {
    'num_layers': 4,
    'kernel_size': [3, 5, 7],
    'dropout': 0.2,
    'dwpw': False,
    'width_mult': 1,
}
encoder = Lipreading(
    modality='video',
    num_classes=500,
    tcn_options=tcn_options,
    backbone_type='resnet',
    relu_type='swish',
    width_mult=1.0,
    use_boundary=False,
    extract_feats=True,
)
official_checkpoint = load_torch_file(checkpoint_path)
encoder.load_state_dict(
    official_checkpoint.get('model_state_dict', official_checkpoint), strict=True
)
head = BiGRUClassifier(len(vocab))

model = PartialFineTuneLipReader(encoder, head).to(device)
assert hasattr(model.encoder.trunk, 'layer4')
print('Model is ready.')

## 6. Frozen feature cache and validation metrics


In [ ]:
class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


def compute_metrics(labels, predictions):
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(
            labels, predictions, average='macro', zero_division=0
        ),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'predicted_classes': len(set(predictions)),
    }


def evaluate_head(head, loader, criterion):
    head.eval()
    loss_sum = 0.0
    labels = []
    predictions = []
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = head(features)
            loss_sum += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = compute_metrics(labels, predictions)
    metrics['loss'] = loss_sum / len(labels)
    return metrics


feature_root = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
train_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_features.pt'
val_cache_path = feature_root / 'dataset_v3_val_lrw_resnet18_features.pt'
if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Frozen feature cache is missing. Run the dataset v3 LRW notebook first.'
    )
train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)
criterion = nn.CrossEntropyLoss()


## 7. Cross-speaker comparison: original vs original + horizontal flip

We hold out `spk03`, `spk17`, and `spk19` one at a time. Each speaker has
examples of all 10 target words. For every fold and seed, both conditions use
the same frozen visual encoder, BiGRU head, optimizer, and stopping rule.

In [ ]:
from sklearn.metrics import confusion_matrix, recall_score


FOLD_SPEAKERS = ['spk03', 'spk17', 'spk19']
SEEDS = [42, 123, 2026]

assert list(train_cache['words']) == train_df['word'].tolist()
for speaker_id in FOLD_SPEAKERS:
    speaker_words = set(train_df.loc[train_df['speaker_id'] == speaker_id, 'word'])
    assert speaker_words == set(vocab), f'{speaker_id} does not contain all classes'


class HorizontalFlipTrainDataset(EndToEndVideoDataset):
    def __getitem__(self, index):
        video, label = super().__getitem__(index)
        return torch.flip(video, dims=[3]), label


# Restore and freeze the official LRW encoder. The flipped cache is created
# only once and reused in future notebook runs.
model.encoder.load_state_dict(
    official_checkpoint.get('model_state_dict', official_checkpoint), strict=True
)
for parameter in model.encoder.parameters():
    parameter.requires_grad = False
model.encoder.eval()

hflip_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_hflip_features.pt'
if hflip_cache_path.exists():
    hflip_cache = load_torch_file(hflip_cache_path)
    flip_features = hflip_cache['features'].float()
    flip_words = list(hflip_cache['words'])
    print('Loaded flipped feature cache:', hflip_cache_path)
else:
    flip_loader = DataLoader(
        HorizontalFlipTrainDataset(train_df, data_root),
        batch_size=16,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    feature_batches = []
    flip_words = []
    with torch.inference_mode():
        for step, (videos, targets) in enumerate(flip_loader, 1):
            videos = videos.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=True):
                features = model.encoder(
                    videos, lengths=[videos.shape[2]] * videos.shape[0]
                )
            feature_batches.append(features.float().cpu())
            flip_words.extend(vocab[index] for index in targets.tolist())
            if step % 25 == 0 or step == len(flip_loader):
                print(f'feature extraction: {step}/{len(flip_loader)}', flush=True)
    flip_features = torch.cat(feature_batches)
    hflip_cache = {'features': flip_features, 'words': flip_words}
    torch.save(hflip_cache, hflip_cache_path)
    print('Saved flipped feature cache:', hflip_cache_path)

assert flip_features.shape == train_cache['features'].shape
assert flip_words == train_df['word'].tolist()


def subset_cache(indices, include_hflip=False):
    indices = np.asarray(indices, dtype=np.int64)
    original_features = train_cache['features'][indices].float()
    original_words = train_df.iloc[indices]['word'].tolist()
    if not include_hflip:
        return {'features': original_features, 'words': original_words}
    return {
        'features': torch.cat([original_features, flip_features[indices]], dim=0),
        'words': original_words + original_words,
    }


def evaluate_with_details(head, loader):
    head.eval()
    labels = []
    predictions = []
    loss_sum = 0.0
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = head(features)
            loss_sum += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return {
        **compute_metrics(labels, predictions),
        'loss': loss_sum / len(labels),
        'labels': labels,
        'predictions': predictions,
    }


def train_fold_condition(fold_speaker, seed, include_hflip):
    set_reproducible_seed(seed)
    is_validation = train_df['speaker_id'].eq(fold_speaker).to_numpy()
    train_indices = np.flatnonzero(~is_validation)
    validation_indices = np.flatnonzero(is_validation)

    fold_train = FeatureSequenceDataset(
        subset_cache(train_indices, include_hflip=include_hflip)
    )
    fold_validation = FeatureSequenceDataset(subset_cache(validation_indices))
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        fold_train,
        batch_size=32,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    validation_loader = DataLoader(
        fold_validation, batch_size=32, shuffle=False, num_workers=0
    )

    head = BiGRUClassifier(len(vocab)).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_f1 = -1.0
    best_loss = float('inf')
    best_epoch = 0
    best_state = None
    without_improvement = 0
    started = time.perf_counter()

    for epoch in range(1, 31):
        head.train()
        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(features), targets)
            loss.backward()
            optimizer.step()

        metrics = evaluate_head(head, validation_loader, criterion)
        improved = (
            metrics['macro_f1'] > best_f1
            or (metrics['macro_f1'] == best_f1 and metrics['loss'] < best_loss)
        )
        if improved:
            best_f1 = metrics['macro_f1']
            best_loss = metrics['loss']
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in head.state_dict().items()
            }
            without_improvement = 0
        else:
            without_improvement += 1
        if without_improvement >= 7:
            break

    head.load_state_dict(best_state)
    details = evaluate_with_details(head, validation_loader)
    condition = 'original+hflip' if include_hflip else 'original'
    result = {
        'fold_speaker': fold_speaker,
        'condition': condition,
        'seed': seed,
        'train_examples': len(fold_train),
        'validation_examples': len(fold_validation),
        'best_epoch': best_epoch,
        'epochs_completed': epoch,
        'validation_loss': details['loss'],
        'validation_accuracy': details['accuracy'],
        'validation_macro_f1': details['macro_f1'],
        'validation_balanced_accuracy': details['balanced_accuracy'],
        'predicted_classes': details['predicted_classes'],
        'training_seconds': time.perf_counter() - started,
    }
    per_class = recall_score(
        details['labels'],
        details['predictions'],
        labels=list(range(len(vocab))),
        average=None,
        zero_division=0,
    )
    recalls = [
        {
            'fold_speaker': fold_speaker,
            'condition': condition,
            'seed': seed,
            'word': word,
            'recall': float(per_class[class_id]),
        }
        for class_id, word in enumerate(vocab)
    ]
    matrix = confusion_matrix(
        details['labels'], details['predictions'], labels=list(range(len(vocab)))
    )
    return result, recalls, best_state, matrix


results = []
per_class_results = []
best_heads = {}
confusion_matrices = {}

for fold_speaker in FOLD_SPEAKERS:
    print(f'\n=== held-out speaker: {fold_speaker} ===')
    for seed in SEEDS:
        for include_hflip in [False, True]:
            result, recalls, state, matrix = train_fold_condition(
                fold_speaker, seed, include_hflip
            )
            results.append(result)
            per_class_results.extend(recalls)
            key = f"{fold_speaker}_{result['condition']}_seed{seed}"
            best_heads[key] = state
            confusion_matrices[key] = matrix
            print(
                f"{result['condition']:14s} seed={seed}: "
                f"F1={result['validation_macro_f1']:.4f}, "
                f"acc={result['validation_accuracy']:.4f}, "
                f"epoch={result['best_epoch']}",
                flush=True,
            )

results_df = pd.DataFrame(results)
per_class_df = pd.DataFrame(per_class_results)
results_df.to_csv(output_dir / 'cross_speaker_hflip_runs.csv', index=False)
per_class_df.to_csv(output_dir / 'cross_speaker_hflip_per_class_recall.csv', index=False)
torch.save(
    {
        'results': results,
        'heads': best_heads,
        'confusion_matrices': confusion_matrices,
        'vocab': vocab,
        'fold_speakers': FOLD_SPEAKERS,
        'test_used': False,
    },
    output_dir / 'cross_speaker_hflip_heads.pt',
)

paired = results_df.pivot_table(
    index=['fold_speaker', 'seed'],
    columns='condition',
    values='validation_macro_f1',
).reset_index()
paired['delta_hflip_minus_original'] = paired['original+hflip'] - paired['original']
summary = {
    'folds': len(FOLD_SPEAKERS),
    'seeds_per_fold': len(SEEDS),
    'original_macro_f1_mean': float(paired['original'].mean()),
    'hflip_macro_f1_mean': float(paired['original+hflip'].mean()),
    'paired_delta_mean': float(paired['delta_hflip_minus_original'].mean()),
    'paired_delta_std': float(paired['delta_hflip_minus_original'].std(ddof=1)),
    'hflip_wins': int((paired['delta_hflip_minus_original'] > 0).sum()),
    'comparisons': len(paired),
}
paired.to_csv(output_dir / 'cross_speaker_hflip_paired.csv', index=False)
pd.DataFrame([summary]).to_csv(
    output_dir / 'cross_speaker_hflip_summary.csv', index=False
)

print('\nPer-run results:')
display(results_df)
print('\nPaired comparison:')
display(paired)
print('\nSummary:', summary)
keep_hflip = summary['paired_delta_mean'] > 0 and summary['hflip_wins'] >= 5
print('DECISION:', 'KEEP HFLIP' if keep_hflip else 'HFLIP NOT CONFIRMED')
print('TEST SPLIT WAS NOT LOADED OR USED.')

## 8. Decision gate

Horizontal flip is confirmed when its mean paired macro-F1 gain is positive
and it wins at least 5 of 9 fold-seed comparisons. A failure here does not
mean the model is broken; it means the augmentation helped `spk06` but did
not generalize consistently enough to become the default protocol.

Do not inspect the final test set during this comparison.